In [ ]:
####IMPORTING MODULES TO WORK
from sklearn.preprocessing import OneHotEncoder,LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split,cross_val_score,GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
import joblib

In [ ]:
####cleaning the data and remove unwanted columns
tit_file=r"Titanic-Dataset.csv"
df=pd.read_csv(tit_file)
feat=df.drop(columns=['PassengerId','Name','Ticket','Cabin'])
feat=feat.drop_duplicates()
target=feat['Survived']
feat=feat.drop(columns=['Survived'])
####Split the data into train and test
X_train,X_test,Y_train,Y_test=train_test_split(feat,target,test_size=0.2,random_state=42)


In [ ]:
####prepare the data  and create models
num_cols = ['Age', 'Fare', 'Pclass', 'Parch', 'SibSp']
cat_cols = ['Sex', 'Embarked']

num_pipe=Pipeline([("Imputer",SimpleImputer(strategy="most_frequent")),
 ("encoder", OneHotEncoder(handle_unknown="ignore"))])

preprosser=ColumnTransformer( transformers=[('num',SimpleImputer(strategy="median"),num_cols),
                                         ("cat",num_pipe,cat_cols)
                                         ])
models={'MLP':MLPClassifier(max_iter=1000),
        'RF':RandomForestClassifier(),
        "GB":GradientBoostingClassifier()}
pipelines={}
for name,model in models.items():
  pipelines[name]=Pipeline(steps=[('prossing',preprosser),
                                  ('model',model)])

In [ ]:
for name,pipe in pipelines.items():
  train_scores=cross_val_score(pipe,X_train,Y_train,cv=5)
  print(name,"train score: ",np.mean(train_scores))

###RF has the best scores
  """MLP train score: MLP train score:  0.7772387096774194
RF train score:  0.7628
GB train score:  0.8012903225806453"""

MLP train score:  0.7772387096774194
RF train score:  0.7628
GB train score:  0.8012903225806453


In [ ]:
grid_pram = {
    'model__n_estimators': [100, 200, 300],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__max_depth': [2, 3, 4],
    'model__min_samples_split': [2, 5, 10]
}
gridcv=GridSearchCV(pipelines['GB'],grid_pram,cv=5)
gridcv.fit(X_train,Y_train)
print(gridcv.best_params_)
print(gridcv.best_score_)
""""{'model__learning_rate': 0.01, 'model__max_depth': 4, 'model__min_samples_split': 5, 'model__n_estimators': 300}
0.8109032258064517
{'model__learning_rate': 0.01, 'model__max_depth': 4, 'model__min_samples_split': 2, 'model__n_estimators': 300}\n0.8092903225806453"""

{'model__learning_rate': 0.01, 'model__max_depth': 4, 'model__min_samples_split': 5, 'model__n_estimators': 300}
0.8109032258064517


"{'model__learning_rate': 0.01, 'model__max_depth': 4, 'model__min_samples_split': 2, 'model__n_estimators': 300}\n0.8092903225806453"

In [29]:
best_model=gridcv.best_estimator_
y_pred=best_model.predict(X_test)
print(accuracy_score(y_pred,Y_test))
joblib.dump(best_model,"titanic_model.pkl")


0.7884615384615384


['titanic_model.pkl']